# SQL - Missing Values

Add scattered, grouped, or mixed `NULL` measurements to a disposable PostgreSQL practice table.

**Instructions:** Use `prepare_data_sql_update.ipynb` (or the corresponding `prepare_data.py`) to recreate the clean `practice_N.weather_hourly` table first. Set `PRACTICE_SCHEMA` below to that exercise's actual schema. Requires a local PostgreSQL server and `psycopg`. This notebook modifies **only the practice table**. Restore clean preparation before rerunning.


In [ ]:
from random import Random
import re

import pandas as pd
import matplotlib.pyplot as plt
import psycopg
from psycopg import sql

# Configuration
DATABASE_NAME = "analytical_workflow_fluency"
PRACTICE_SCHEMA = "practice_014"  # Match the exercise's preparation configuration.
PRACTICE_TABLE = "weather_hourly"
TIME_COLUMN = "time"
KEY_COLUMN = "time"  # A unique row identifier; may differ from the time column.
TIMEZONE = "America/Edmonton"
START_DATE = "2025-08-01"
END_DATE = "2025-09-01"  # Exclusive local calendar date.
SEED = 814205

# Missing Values
MISSING_TARGET_COLUMN = "relative_humidity_2m"
MISSING_METHOD = "mixed"  # "scattered", "grouped", "mixed", or "random".
MISSING_RATE = (0.03, 0.08)  # Share of current eligible nonmissing values changed.
MISSING_GROUPS = 3
MISSING_GROUPED_SHARE = 0.70


In [ ]:
def select_positions(n, rng, *, method, rate, n_groups, grouped_share):
    """Select a fraction of the current eligible records in ordered positions.

    Grouped blocks have at least two consecutive eligible records; mixed
    placement reserves at least one independently scattered selection.
    """
    if n < 1:
        raise ValueError("No eligible records.")
    if method not in ("scattered", "grouped", "mixed", "random"):
        raise ValueError("Unknown placement method.")
    if not 0 < rate[0] <= rate[1] < 1:
        raise ValueError("Rate must be between 0 and 1.")
    if n_groups < 1 or not 0 < grouped_share < 1:
        raise ValueError("Invalid group count or grouped share.")

    chosen = rng.choice(("scattered", "grouped", "mixed")) if method == "random" else method
    count = max(1, round(n * rng.uniform(*rate)))
    if chosen == "mixed" and count < 3:
        raise ValueError("Mixed placement needs two grouped and one scattered record.")
    grouped_count = (count if chosen == "grouped" else
                     max(2, min(count - 1, round(count * grouped_share))) if chosen == "mixed" else 0)
    selected = set()
    if grouped_count:
        if grouped_count < 2:
            raise ValueError("Grouped placement requires at least two affected records.")
        groups = min(n_groups, grouped_count // 2)
        if n - grouped_count < groups - 1:
            raise ValueError("Insufficient eligible records for separate groups.")
        length, extra = divmod(grouped_count, groups)
        lengths = [length + (i < extra) for i in range(groups)]
        rng.shuffle(lengths)
        slack = n - grouped_count - (groups - 1)
        dividers = sorted(rng.sample(range(slack + groups), groups))
        gaps = ([dividers[0]] +
                [dividers[i] - dividers[i - 1] - 1 for i in range(1, groups)] +
                [slack + groups - 1 - dividers[-1]])
        cursor = gaps[0]
        for i, block_length in enumerate(lengths):
            selected.update(range(cursor, cursor + block_length))
            if i < groups - 1:
                cursor += block_length + 1 + gaps[i + 1]

    scattered_count = count - grouped_count
    if scattered_count:
        remaining = [i for i in range(n) if i not in selected]
        selected.update(rng.sample(remaining, k=scattered_count))
    if len(selected) != count:
        raise AssertionError("Selected count differs from the intended count.")
    return sorted(selected), chosen


In [ ]:
def add_missing_values_sql(
    conn, rng, *, schema, table, time_column, key_column, timezone,
    start_date, end_date, target_column, method, missing_rate,
    n_groups, grouped_share,
):
    # Never allow this function to target the canonical schema.
    if not re.fullmatch(r"practice_[0-9]+", schema):
        raise ValueError("Corruption is restricted to practice_N schemas.")
    if target_column in (time_column, key_column):
        raise ValueError("Time and key columns cannot be missing-value targets.")
    if conn.autocommit:
        raise ValueError("Use a transaction, not autocommit.")

    start = pd.Timestamp(start_date, tz=timezone).to_pydatetime()
    end = pd.Timestamp(end_date, tz=timezone).to_pydatetime()
    if start >= end:
        raise ValueError("Start date must precede end date.")

    source = sql.Identifier(schema, table)
    time_id = sql.Identifier(time_column)
    key_id = sql.Identifier(key_column)
    target_id = sql.Identifier(target_column)
    rows = conn.execute(
        sql.SQL("""
            SELECT {key}, ({target} IS NULL)
            FROM {source}
            WHERE {time} >= %s AND {time} < %s
            ORDER BY {time}, {key}
        """).format(time=time_id, key=key_id, target=target_id, source=source),
        (start, end),
    ).fetchall()

    keys = [k for k, _ in rows]
    if any(k is None for k in keys) or len(set(keys)) != len(keys):
        raise ValueError("Expected unique non-NULL row keys; apply this before duplication.")
    if any(is_null for _, is_null in rows):
        raise ValueError("Target already has NULLs in scope. Restore clean practice data first.")

    positions, chosen_method = select_positions(
        len(keys), rng, method=method, rate=missing_rate,
        n_groups=n_groups, grouped_share=grouped_share,
    )
    selected_keys = [keys[i] for i in positions]

    changed = conn.execute(
        sql.SQL("""
            UPDATE {source}
            SET {target} = NULL
            WHERE {key} = ANY(%s)
              AND {time} >= %s AND {time} < %s
              AND {target} IS NOT NULL
            RETURNING {key}
        """).format(source=source, target=target_id, key=key_id, time=time_id),
        (selected_keys, start, end),
    ).fetchall()

    if len(changed) != len(selected_keys) or {r[0] for r in changed} != set(selected_keys):
        raise ValueError("Changed rows did not match selected keys; transaction must roll back.")
    return len(changed), chosen_method


In [ ]:
# Small trial: multiple groups, scattered omissions, reproducible selection.
for kind in ("scattered", "grouped", "mixed", "random"):
    a, chosen = select_positions(
        240, Random(SEED), method=kind, rate=(0.05, 0.05),
        n_groups=3, grouped_share=0.70,
    )
    b, _ = select_positions(
        240, Random(SEED), method=kind, rate=(0.05, 0.05),
        n_groups=3, grouped_share=0.70,
    )
    assert a == b and len(a) == 12 and len(set(a)) == 12
    print(kind, "->", chosen, ":", a)

# Boundaries are local calendar dates, converted for PostgreSQL TIMESTAMPTZ comparisons.
start = pd.Timestamp(START_DATE, tz=TIMEZONE)
end = pd.Timestamp(END_DATE, tz=TIMEZONE)
assert start < end
print("Selected interval (UTC):", start.tz_convert("UTC"), "to", end.tz_convert("UTC"))
for day, expected in (("2025-03-09", 23), ("2025-11-02", 25)):
    a = pd.Timestamp(day, tz=TIMEZONE)
    b = a + pd.DateOffset(days=1)
    assert (b.tz_convert("UTC") - a.tz_convert("UTC")) / pd.Timedelta(hours=1) == expected


In [ ]:
# Verify rollback before committing the actual corruption.
with psycopg.connect(dbname=DATABASE_NAME) as conn:
    source = sql.Identifier(PRACTICE_SCHEMA, PRACTICE_TABLE)
    target_id = sql.Identifier(MISSING_TARGET_COLUMN)
    count_query = sql.SQL("SELECT COUNT(*) FROM {} WHERE {} IS NULL").format(source, target_id)
    before_nulls = conn.execute(count_query).fetchone()[0]
    try:
        with conn.transaction():  # Nested transaction/savepoint after the SELECT.
            add_missing_values_sql(
                conn, Random(SEED),
                schema=PRACTICE_SCHEMA, table=PRACTICE_TABLE,
                time_column=TIME_COLUMN, key_column=KEY_COLUMN, timezone=TIMEZONE,
                start_date=START_DATE, end_date=END_DATE,
                target_column=MISSING_TARGET_COLUMN, method=MISSING_METHOD,
                missing_rate=MISSING_RATE, n_groups=MISSING_GROUPS,
                grouped_share=MISSING_GROUPED_SHARE,
            )
            raise RuntimeError("Intentional rollback test")
    except RuntimeError as exc:
        if str(exc) != "Intentional rollback test":
            raise
    assert conn.execute(count_query).fetchone()[0] == before_nulls
print("Rollback test passed")


In [ ]:
# Apply to the prepared disposable table (read and verify within one transaction).
with psycopg.connect(dbname=DATABASE_NAME) as conn:
    source = sql.Identifier(PRACTICE_SCHEMA, PRACTICE_TABLE)
    time_id = sql.Identifier(TIME_COLUMN)
    target_id = sql.Identifier(MISSING_TARGET_COLUMN)
    start = pd.Timestamp(START_DATE, tz=TIMEZONE).to_pydatetime()
    end = pd.Timestamp(END_DATE, tz=TIMEZONE).to_pydatetime()

    summary_sql = sql.SQL("""
        SELECT COUNT(*), COUNT(*) FILTER (WHERE {target} IS NULL)
        FROM {source}
        WHERE {time} >= %s AND {time} < %s
    """).format(source=source, target=target_id, time=time_id)
    before = conn.execute(summary_sql, (start, end)).fetchone()

    changed, selected_method = add_missing_values_sql(
        conn, Random(SEED),
        schema=PRACTICE_SCHEMA, table=PRACTICE_TABLE,
        time_column=TIME_COLUMN, key_column=KEY_COLUMN, timezone=TIMEZONE,
        start_date=START_DATE, end_date=END_DATE,
        target_column=MISSING_TARGET_COLUMN, method=MISSING_METHOD,
        missing_rate=MISSING_RATE, n_groups=MISSING_GROUPS,
        grouped_share=MISSING_GROUPED_SHARE,
    )

    after = conn.execute(summary_sql, (start, end)).fetchone()
    assert after[0] == before[0] and after[1] == before[1] + changed
    print("Method:", selected_method, "| New NULLs:", changed)
    print("Scoped rows:", before[0], "| NULLs before/after:", before[1], after[1])
# An exception inside the connection context rolls back the whole transaction.


In [ ]:
# Inspect: number of NULL measurements and affected timestamps.
with psycopg.connect(dbname=DATABASE_NAME) as conn:
    source = sql.Identifier(PRACTICE_SCHEMA, PRACTICE_TABLE)
    time_id = sql.Identifier(TIME_COLUMN)
    target_id = sql.Identifier(MISSING_TARGET_COLUMN)
    missing_times = conn.execute(
        sql.SQL("""
            SELECT {time}
            FROM {source}
            WHERE {time} >= %s AND {time} < %s
              AND {target} IS NULL
            ORDER BY {time}
        """).format(time=time_id, source=source, target=target_id),
        (pd.Timestamp(START_DATE, tz=TIMEZONE).to_pydatetime(),
         pd.Timestamp(END_DATE, tz=TIMEZONE).to_pydatetime()),
    ).fetchall()

missing_df = pd.DataFrame(missing_times, columns=[TIME_COLUMN])
missing_df.head(20)


In [ ]:
# Visualize missing measurements by local reporting date.
local_date = pd.to_datetime(missing_df[TIME_COLUMN], utc=True).dt.tz_convert(TIMEZONE).dt.date
counts = local_date.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(counts.index.astype(str), counts.values)
ax.set(title=f"Missing {MISSING_TARGET_COLUMN} measurements by date", xlabel="Local date", ylabel="Missing values")
step = max(1, len(counts) // 8)
ax.set_xticks(range(0, len(counts), step), [str(d) for d in counts.index[::step]], rotation=45, ha="right")
fig.tight_layout()
plt.show()
